In [19]:
import pandas as pd
scores = pd.read_csv("results_scored.csv")


In [20]:
scores.shape

(10783, 10)

In [22]:
scores = scores.drop_duplicates()
scores.shape

(4841, 10)

In [67]:
duplicates = scores[
    scores.duplicated(
        subset=['Query', 'Method', 'Rank'],
        keep=False
    )
].sort_values(['Query', 'Method', 'Rank'])


In [68]:
scores = scores.drop_duplicates(
    subset=['Query', 'Method', 'Rank'],
    keep='first'
)

In [69]:
scores.shape

# This is what we needed

(1680, 12)

-  `human_scores` - is the results from human annotators
- `human_seen_llm_results` - is the results scored by the llm in the pairwise comparisons 

In [79]:
human_scores = pd.read_csv("human_scores.csv")
human_scores

,Unnamed: 0,Query,Query Category,Method,Similarity Score (%),numbered_result,Text,Psalm Num,Verse,User,Score
0,0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,26.10,1,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,25.67,2,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,21.90,3,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.33,4,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.12,5,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...,...,...
855,941,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p09,8
856,942,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p03,1
857,943,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p10,2
858,944,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7


In [86]:
human_scores = human_scores.rename(columns={'numbered_result':'Rank'})

In [82]:
human_seen_results = pd.read_csv("results_from_humans_scored.csv")
#human_seen_results

In [92]:
human_seen_llm_results = human_seen_results.drop_duplicates(
    subset=['Query', 'Method', 'Rank'],
    keep='first'
)

human_seen_llm_results = human_seen_llm_results.sort_values(
    by='Query'
)

human_seen_llm_results

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score
79,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,False,A,3
80,Create in me a clean heart,BERT,3,68.69,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",False,B,1
81,Create in me a clean heart,SBERT,3,95.62,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,False,C,2
82,Create in me a clean heart,TFIDF,3,14.20,Psalter,23,"The earth is the Lord's, and the fulness there...",False,D,0
102,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,False,A,2
...,...,...,...,...,...,...,...,...,...,...
250,protection from enemies,TFIDF,3,0.00,Bible,103,By David Bless the Lord O my soul O Lord my Go...,False,D,0
322,protection from enemies,TFIDF_GLoVe,4,22.61,Psalter,28,"Bring unto the Lord, O ye sons of God, bring u...",False,A,1
323,protection from enemies,BERT,4,99.57,Psalter,42,"Judge me, God, and give judgment in my cause a...",False,B,0
324,protection from enemies,SBERT,4,96.26,Bible,53,For the End in hymns concerning understanding ...,False,C,3


In [93]:
common_queries = list(
    set(human_scores['Query'].unique()) &
    set(human_seen_results['Query'].unique())
)
common_queries

['Rejoice, O ye heavens, sound the trumpets, ye foundation of the earth, thunder forth gladness, O ye mountains: for behold, Emmanuel to the Cross our sins, and the Giver of Life hath slain death, raising up Adam; for He loveth mankind.',
 'protection from enemies',
 'How does the psalmist express trust in God while surrounded by fear and uncertainty?',
 'Have mercy on me, O God, have mercy on me. For my soul trusts in Thee, and in the shadow of Thy wings will I hope, until iniquity pass away.',
 'Create in me a clean heart',
 'mercy',
 'For the Peace of the world',
 'The Lord is my shepherd',
 'prayer',
 'Verses where the psalmist remembers past deliverance and uses it to find hope in present trials.',
 'praise in times of suffering']

In [94]:
keys = ['Query', 'Method', 'Rank']

# Make sure there is only one LLM score per result
llm_scores = human_seen_llm_results[
    keys + ['Score']
].drop_duplicates(
    subset=keys,
    keep='first'
)

# Add LLM score to the human scores
human_scores_combined = human_scores.merge(
    llm_scores,
    on=keys,
    how='left',
    validate='one_to_one'
)

# Rename the LLM score
human_scores_combined = human_scores_combined.rename(
    columns={'Score': 'LLM_Score'}
)

print("Human scores:", len(human_scores))
print("Combined:", len(human_scores_combined))

MergeError: Merge keys are not unique in left dataset; not a one-to-one merge
Duplicates in left:
                      Query      Method  Rank
Create in me a clean heart TFIDF_GLoVe     1
Create in me a clean heart TFIDF_GLoVe     1
Create in me a clean heart TFIDF_GLoVe     1
Create in me a clean heart TFIDF_GLoVe     2
Create in me a clean heart TFIDF_GLoVe     2 ...

## Looking at overlap

In [87]:
import numpy as np 
common = human_scores.merge(
    human_seen_llm_results,
    on=['Query', 'Method', 'Rank'],
    how='inner'
)

common

,Unnamed: 0,Query,Query Category,Method,Similarity Score (%)_x,Rank,Text_x,Psalm Num_x,Verse_x,User,Score_x,Similarity Score (%)_y,Text_y,Psalm Num_y,Verse_y,Scored,Letter,Score_y
0,0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,26.10,1,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,False,A,3
1,1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,25.67,2,Bible,4,For the End in psalms an ode by David You hear...,caden,6,25.67,Bible,4,For the End in psalms an ode by David You hear...,False,A,2
2,2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,21.90,3,Bible,31,By David concerning understanding Blessed are ...,caden,3,21.90,Bible,31,By David concerning understanding Blessed are ...,False,A,3
3,3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.33,4,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,False,A,1
4,4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.12,5,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,False,A,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
847,941,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p09,8,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...,False,D,0
848,942,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p03,1,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...,False,D,0
849,943,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p10,2,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",False,D,1
850,944,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",False,D,1


In [34]:
keys = ['Query', 'Method', 'Rank']

human_seen_results_clean = human_results.merge(
    human_seen_results[keys].drop_duplicates(),
    on=keys,
    how='inner'
)

print("Before:", len(human_seen_results))
print("After:", len(human_seen_results_clean))

Before: 765
After: 215


In [35]:
human_seen_results_clean

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...
...,...,...,...,...,...,...,...
210,Verses where the psalmist remembers past deliv...,TFIDF,1,16.33,Psalter,86,His foundations are in the holy mountains. The...
211,Verses where the psalmist remembers past deliv...,TFIDF,2,9.98,Psalter,23,"The earth is the Lord's, and the fulness there..."
212,Verses where the psalmist remembers past deliv...,TFIDF,3,7.79,Bible,130,1An ode of ascents by David OLord My heart is ...
213,Verses where the psalmist remembers past deliv...,TFIDF,4,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...


### Adding the scores from the LLM

In [36]:
keys = ['Query', 'Method', 'Rank']

# One row per result
scores_to_keep = (
    human_seen_results
    .drop_duplicates(subset=keys, keep='last')
)

# Merge scores onto the 215-row master dataset
human_results_scored = human_results.merge(
    scores_to_keep,
    on=keys,
    how='left',
    suffixes=('', '_seen')
)

print(len(human_results_scored))  # Should be 215

215


In [66]:
human_results_scored.head(25)

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Similarity Score (%)_seen,Text_seen,Psalm Num_seen,Verse_seen,Scored,Letter,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,False,A,3
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,25.67,Bible,4,For the End in psalms an ode by David You hear...,True,A,0
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,21.90,Bible,31,By David concerning understanding Blessed are ...,True,A,3
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,True,A,3
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,False,A,3
5,Create in me a clean heart,BERT,1,69.89,Bible,18,For the End a psalm by David The heavens decla...,69.89,Bible,18,For the End a psalm by David The heavens decla...,False,B,2
6,Create in me a clean heart,BERT,2,69.06,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",69.06,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",True,B,1
7,Create in me a clean heart,BERT,3,68.69,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",68.69,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",True,B,0
8,Create in me a clean heart,BERT,4,68.66,Psalter,43,"We have heard with our ears, O God, for our fa...",68.66,Psalter,43,"We have heard with our ears, O God, for our fa...",True,B,2
9,Create in me a clean heart,BERT,5,68.31,Bible,147,Alleluia of Aggeus and Zacharias Praise the Lo...,68.31,Bible,147,Alleluia of Aggeus and Zacharias Praise the Lo...,False,B,2


In [38]:
draft_scores = scores.copy()

draft_scores.shape

(1610, 10)

In [39]:
# 3. Remove the old versions of these 215 results from scores
scores_without_human = (
    draft_scores
    .merge(
        human_results[keys],
        on=keys,
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns='_merge')
)


In [40]:
# 4. Add the newly scored versions
scores_updated = pd.concat(
    [scores_without_human, human_results_scored],
    ignore_index=True
)

print("Original scores:", len(draft_scores))
print("Updated scores:", len(scores_updated))
print("Rows replaced:", len(human_results_scored))

Original scores: 1610
Updated scores: 1718
Rows replaced: 215


In [41]:
scores_updated

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score,Similarity Score (%)_seen,Text_seen,Psalm Num_seen,Verse_seen
0,struggle,BERT,4,31.49,Psalter,142,"Hear my prayer, O Lord; give ear unto my suppl...",False,A,1,NaN,NaN,NaN,NaN
1,struggle,SBERT,4,96.08,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,False,B,0,NaN,NaN,NaN,NaN
2,creation,TF-IDF + GloVe,2,13.98,Bible,122,1An ode of ascents Ilift my eyes to You Whodwe...,False,A,2,NaN,NaN,NaN,NaN
3,creation,BERT,2,40.51,Psalter,42,"Judge me, God, and give judgment in my cause a...",False,B,1,NaN,NaN,NaN,NaN
4,creation,SBERT,2,96.40,Psalter,146,The Lord doth build up Jerusalem; He shall gat...,False,C,3,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1713,Verses where the psalmist remembers past deliv...,TFIDF,1,16.33,Psalter,86,His foundations are in the holy mountains. The...,True,D,0,16.33,Psalter,86.0,His foundations are in the holy mountains. The...
1714,Verses where the psalmist remembers past deliv...,TFIDF,2,9.98,Psalter,23,"The earth is the Lord's, and the fulness there...",False,D,2,9.98,Psalter,23.0,"The earth is the Lord's, and the fulness there..."
1715,Verses where the psalmist remembers past deliv...,TFIDF,3,7.79,Bible,130,1An ode of ascents by David OLord My heart is ...,False,D,1,7.79,Bible,130.0,1An ode of ascents by David OLord My heart is ...
1716,Verses where the psalmist remembers past deliv...,TFIDF,4,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...,True,D,3,7.08,Psalter,61.0,Shall not my soul be subject unto God? for fro...


In [42]:
scores = draft_scores

In [43]:
scores

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score
0,struggle,BERT,4,31.49,Psalter,142,"Hear my prayer, O Lord; give ear unto my suppl...",False,A,1
1,struggle,SBERT,4,96.08,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,False,B,0
2,creation,TF-IDF + GloVe,2,13.98,Bible,122,1An ode of ascents Ilift my eyes to You Whodwe...,False,A,2
3,creation,BERT,2,40.51,Psalter,42,"Judge me, God, and give judgment in my cause a...",False,B,1
4,creation,SBERT,2,96.40,Psalter,146,The Lord doth build up Jerusalem; He shall gat...,False,C,3
...,...,...,...,...,...,...,...,...,...,...
5052,Which psalms use parallelism heavily to emphas...,TF-IDF,1,19.30,Psalter,86,His foundations are in the holy mountains. The...,False,D,0
5151,protection from enemies,TF-IDF + GloVe,2,13.98,Bible,122,1An ode of ascents Ilift my eyes to You Whodwe...,False,A,2
5152,protection from enemies,BERT,2,53.39,Psalter,42,"Judge me, God, and give judgment in my cause a...",False,B,1
5153,protection from enemies,SBERT,2,95.14,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,False,C,3


### Data Cleaning 
There were a lot of rowsa that came in a duplicates. Herewe are fixing that to ensure that we are seeing only one record for each Query, Method, and Rank combination.  

In [44]:
scores_clean = scores.drop_duplicates(
    subset=[
        "Query",
        "Method",
        "Rank",
        "Similarity Score (%)",
        "Text",
        "Psalm Num",
        "Verse",
        "Letter"
    ],
    keep="first"
)

scores = scores_clean
print(scores.shape)

scores[scores['Query'] == 'struggle'].drop_duplicates()

(1610, 10)


,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score
0,struggle,BERT,4,31.49,Psalter,142,"Hear my prayer, O Lord; give ear unto my suppl...",False,A,1
1,struggle,SBERT,4,96.08,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,False,B,0
33,struggle,BERT,5,30.93,Psalter,1,Blessed is the man that hath not walked in the...,False,A,0
34,struggle,SBERT,5,95.96,Psalter,146,The Lord doth build up Jerusalem; He shall gat...,False,B,1
406,struggle,BERT,2,34.15,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",False,A,1
407,struggle,SBERT,2,96.15,Psalter,149,"Sing unto the Lord a new song, His praise is i...",False,B,0
1653,struggle,BERT,1,35.08,Psalter,42,"Judge me, God, and give judgment in my cause a...",False,A,0
1654,struggle,SBERT,1,96.24,Psalter,150,Praise God in His holy ones; praise Him in the...,False,B,1
1655,struggle,TF-IDF,1,0.00,Psalter,150,Praise God in His holy ones; praise Him in the...,False,C,2
2115,struggle,BERT,3,31.66,Bible,18,For the End a psalm by David The heavens decla...,False,A,1


## Missing result Imputation and Score Rellocation
Algorithm
----
    1. groupby 'Query' and 'Rank"
    2. check if all four methods are recorded
        - 'BERT', 'SBERT', 'TF-IDF + GloVe', 'TF-IDF'
        - record the number of missing methods
    3. if there is at least one method not being represented
        1. add rows for the missing methods with giving the score as 1
        2. increase the other method's score with the number of previously missing methods
        3. ensure thatno method has a score higher than 3
    5. save to a new Dataframe 

In [45]:
import pandas as pd

EXPECTED_METHODS = {
    "BERT",
    "SBERT",
    "TF-IDF + GloVe",
    "TF-IDF"
}

new_groups = []
augmented_scores = pd.DataFrame()

for (query, rank), group in scores.groupby(["Query", "Rank"]):

    group = group.copy()

    # Mark original rows
    group["Imputed"] = False

    present_methods = set(group["Method"])
    missing_methods = EXPECTED_METHODS - present_methods
    n_missing = len(missing_methods)

    if n_missing > 0:

        # Increase existing scores
        group["Score"] = (group["Score"] + n_missing).clip(upper=3)

        for method in missing_methods:

            new_row = {col: pd.NA for col in scores.columns}

            # Required fields
            new_row["Query"] = query
            new_row["Rank"] = rank
            new_row["Method"] = method
            new_row["Score"] = 0

            # Explicitly null out retrieval data
            new_row["Similarity Score (%)"] = pd.NA
            new_row["Text"] = pd.NA
            new_row["Psalm Num"] = pd.NA
            new_row["Verse"] = pd.NA
            new_row["Scored"] = pd.NA
            new_row["Letter"] = pd.NA

            new_row["Imputed"] = True

            group = pd.concat(
                [group, pd.DataFrame([new_row])],
                ignore_index=True
            )

    new_groups.append(group)

augmented_scores = pd.concat(new_groups, ignore_index=True)

# Verification
print(
    augmented_scores
    .groupby(["Query", "Rank"])["Method"]
    .nunique()
    .value_counts()
)

print(
    augmented_scores["Imputed"]
    .value_counts()
)

Method
4    420
Name: count, dtype: int64
Imputed
False    1610
True       70
Name: count, dtype: int64


In [46]:
augmented_scores[augmented_scores["Query"] == 'struggle']

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score,Imputed
1600,struggle,BERT,1,35.08,Psalter,42,"Judge me, God, and give judgment in my cause a...",False,A,1,False
1601,struggle,SBERT,1,96.24,Psalter,150,Praise God in His holy ones; praise Him in the...,False,B,2,False
1602,struggle,TF-IDF,1,0.0,Psalter,150,Praise God in His holy ones; praise Him in the...,False,C,3,False
1603,struggle,TF-IDF + GloVe,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,True
1604,struggle,BERT,2,34.15,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",False,A,3,False
1605,struggle,SBERT,2,96.15,Psalter,149,"Sing unto the Lord a new song, His praise is i...",False,B,2,False
1606,struggle,TF-IDF + GloVe,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,True
1607,struggle,TF-IDF,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,True
1608,struggle,BERT,3,31.66,Bible,18,For the End a psalm by David The heavens decla...,False,A,3,False
1609,struggle,SBERT,3,96.1,Psalter,147,"Praise the Lord, O Jerusalem; praise thy God, ...",False,B,2,False


In [48]:
query = "struggle"

orig = scores[scores["Query"] == query]
aug = augmented_scores[augmented_scores["Query"] == query]

print(f"Query: '{query}'")
print("-" * 40)
print(f"Original rows:              {len(orig)}")
print(f"Original unique rows:       {len(orig.drop_duplicates())}")
print(f"Original duplicates:        {len(orig) - len(orig.drop_duplicates())}")
print()
print(f"Augmented rows:             {len(aug)}")
print(f"Augmented unique rows:      {len(aug.drop_duplicates())}")
print(f"Augmented duplicates:       {len(aug) - len(aug.drop_duplicates())}")

Query: 'struggle'
----------------------------------------
Original rows:              11
Original unique rows:       11
Original duplicates:        0

Augmented rows:             20
Augmented unique rows:      20
Augmented duplicates:       0


In [49]:
# Saving new Data as scores
scores = augmented_scores

In [64]:
pd.merge(scores, human_scores,  how='inner')

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score,...,p04,p05,p06,p07,p08,p09,p10,p13,p16,p17
0,Create in me a clean heart,BERT,1,69.89,Bible,18,For the End a psalm by David The heavens decla...,False,B,1,...,NaN,NaN,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Create in me a clean heart,BERT,2,69.06,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",False,B,0,...,NaN,NaN,5.0,NaN,7.0,NaN,4.0,NaN,NaN,NaN
2,Create in me a clean heart,BERT,3,68.69,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",False,B,1,...,NaN,9.0,7.0,NaN,NaN,NaN,NaN,10.0,NaN,NaN
3,Create in me a clean heart,BERT,4,68.66,Psalter,43,"We have heard with our ears, O God, for our fa...",False,B,2,...,NaN,7.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Create in me a clean heart,BERT,5,68.31,Bible,147,Alleluia of Aggeus and Zacharias Praise the Lo...,False,B,2,...,NaN,6.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
5,prayer,BERT,1,43.82,Bible,18,For the End a psalm by David The heavens decla...,False,B,0,...,NaN,6.0,NaN,NaN,NaN,NaN,3.0,9.0,NaN,NaN
6,prayer,BERT,2,40.30,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",False,B,1,...,NaN,NaN,NaN,NaN,NaN,7.0,2.0,NaN,NaN,NaN
7,prayer,BERT,3,39.39,Psalter,42,"Judge me, God, and give judgment in my cause a...",False,B,0,...,NaN,8.0,6.0,NaN,NaN,NaN,NaN,10.0,NaN,NaN
8,prayer,BERT,4,38.91,Psalter,149,"Sing unto the Lord a new song, His praise is i...",False,B,3,...,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN
9,prayer,BERT,5,38.70,Psalter,53,"Save me, O God, by Thy name, and judge me by T...",False,B,3,...,NaN,10.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Adding in the Query Category


In [50]:
queries = pd.read_csv('queries.csv')

In [51]:
scores["Query Category"] = scores["Query"].map(
    queries.set_index("Query")["Query Category"]
)

scores.head(1)

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score,Imputed,Query Category
0,How does the psalmist describe God's compassi...,TF-IDF + GloVe,1,16.0,Bible,79,1For the End concerning things that shall be c...,False,A,0,False,Long / Complex


In [52]:
queries['Query Category'].value_counts().sort_values(ascending=False)

Query Category
Orthodox Serivce Quotes    20
Short Keyword              20
Phrase / Exact Match       16
Thematic / Semantic        15
Long / Complex             13
Name: count, dtype: int64

In [53]:
# Adding the 5 results per the four methods for each query query to make sure it 
# matches up with the data in the scores dataframe

(queries['Query Category'].value_counts()*20).sort_values(ascending=False)

Query Category
Orthodox Serivce Quotes    400
Short Keyword              400
Phrase / Exact Match       320
Thematic / Semantic        300
Long / Complex             260
Name: count, dtype: int64

In [54]:
#pivot for each category
pd.pivot_table(
    data=scores,
    index='Query Category',
    values='Score',
    aggfunc='count'
).sort_values('Score', ascending=False)

,Score
Query Category,
Orthodox Serivce Quotes,400
Short Keyword,400
Phrase / Exact Match,320
Thematic / Semantic,300
Long / Complex,260


* we can confirm that alll of the query caetorgiers are properly represented within the data based on the queries used and the corresponding query categories
* by working and adjusting the math for the values counts we can see that all the data is represented correctly

============================================

## Krippendorff Alphs from Human Annotators and LLM

In [55]:
scores

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score,Imputed,Query Category
0,How does the psalmist describe God's compassi...,TF-IDF + GloVe,1,16.0,Bible,79,1For the End concerning things that shall be c...,False,A,0,False,Long / Complex
1,How does the psalmist describe God's compassi...,BERT,1,75.36,Bible,18,For the End a psalm by David The heavens decla...,False,B,3,False,Long / Complex
2,How does the psalmist describe God's compassi...,SBERT,1,97.46,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,False,C,2,False,Long / Complex
3,How does the psalmist describe God's compassi...,TF-IDF,1,14.64,Psalter,84,"Lord, Thou hast been favourable unto Thy land;...",False,D,1,False,Long / Complex
4,How does the psalmist describe God's compassi...,TF-IDF + GloVe,2,13.98,Bible,122,1An ode of ascents Ilift my eyes to You Whodwe...,False,A,2,False,Long / Complex
...,...,...,...,...,...,...,...,...,...,...,...,...
1675,wisdom,TF-IDF,4,9.48,Psalter,110,"I will give Thee thanks, O Lord, with my whole...",False,D,0,False,Short Keyword
1676,wisdom,TF-IDF + GloVe,5,12.74,Bible,11,For the End concerning the eighth a psalm by D...,False,A,3,False,Short Keyword
1677,wisdom,BERT,5,34.61,Bible,36,Of David Do not be envious of those who do evi...,False,B,0,False,Short Keyword
1678,wisdom,SBERT,5,96.46,Psalter,149,"Sing unto the Lord a new song, His praise is i...",False,C,1,False,Short Keyword


In [56]:
human_scores = pd.read_csv("human_scores.csv")
human_scores

,Unnamed: 0,Query,Query Category,Method,Similarity Score (%),numbered_result,Text,Psalm Num,Verse,User,Score
0,0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,26.10,1,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,25.67,2,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,21.90,3,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.33,4,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.12,5,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...,...,...
855,941,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p09,8
856,942,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p03,1
857,943,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p10,2
858,944,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7


In [57]:
print(human_seen_results.columns.tolist())

['Query', 'Method', 'Rank', 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'Scored', 'Letter', 'Score']


In [59]:
human_scores = human_scores.rename(columns={'numbered_result':'Rank'})

In [60]:
keys = [
    'Query',
    'Method',
    'Rank',
    'Similarity Score (%)',
    'Text',
    'Psalm Num',
    'Verse'
]

# Turn the repeated human evaluations into 4 score columns
human_scores = (
    human_scores
    .pivot_table(
        index=keys,
        columns='User',
        values='Score',
        aggfunc='first'
    )
    .reset_index()
)

human_scores.columns.name = None

In [61]:
human_scores

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,caden,p01,p02,...,p04,p05,p06,p07,p08,p09,p10,p13,p16,p17
0,Create in me a clean heart,BERT,1,69.89,Bible,18,For the End a psalm by David The heavens decla...,4.0,NaN,8.0,...,NaN,NaN,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Create in me a clean heart,BERT,2,69.06,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",8.0,NaN,NaN,...,NaN,NaN,5.0,NaN,7.0,NaN,4.0,NaN,NaN,NaN
2,Create in me a clean heart,BERT,3,68.69,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",5.0,NaN,NaN,...,NaN,9.0,7.0,NaN,NaN,NaN,NaN,10.0,NaN,NaN
3,Create in me a clean heart,BERT,4,68.66,Psalter,43,"We have heard with our ears, O God, for our fa...",0.0,3.0,NaN,...,NaN,7.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Create in me a clean heart,BERT,5,68.31,Bible,147,Alleluia of Aggeus and Zacharias Praise the Lo...,5.0,NaN,NaN,...,NaN,6.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,protection from enemies,TFIDF_GLoVe,1,25.38,Bible,3,A psalm by David when he fled from the face of...,10.0,NaN,10.0,...,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
211,protection from enemies,TFIDF_GLoVe,2,25.01,Bible,28,A psalm by David the final day of the Feast of...,9.0,3.0,NaN,...,NaN,NaN,0.0,NaN,8.0,NaN,NaN,NaN,NaN,NaN
212,protection from enemies,TFIDF_GLoVe,3,23.85,Psalter,86,His foundations are in the holy mountains. The...,4.0,NaN,NaN,...,5.0,NaN,0.0,NaN,5.0,NaN,NaN,NaN,NaN,NaN
213,protection from enemies,TFIDF_GLoVe,4,22.61,Psalter,28,"Bring unto the Lord, O ye sons of God, bring u...",3.0,NaN,NaN,...,NaN,NaN,1.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN


In [62]:
human_scores['Similarity Score (%)'] = pd.to_numeric(
    human_scores['Similarity Score (%)'],
    errors='coerce'
)

scores['Similarity Score (%)'] = pd.to_numeric(
    scores['Similarity Score (%)'],
    errors='coerce'
)

In [63]:
pd.merge(scores, human_scores, how='inner')

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score,...,p04,p05,p06,p07,p08,p09,p10,p13,p16,p17
0,Create in me a clean heart,BERT,1,69.89,Bible,18,For the End a psalm by David The heavens decla...,False,B,1,...,NaN,NaN,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Create in me a clean heart,BERT,2,69.06,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",False,B,0,...,NaN,NaN,5.0,NaN,7.0,NaN,4.0,NaN,NaN,NaN
2,Create in me a clean heart,BERT,3,68.69,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",False,B,1,...,NaN,9.0,7.0,NaN,NaN,NaN,NaN,10.0,NaN,NaN
3,Create in me a clean heart,BERT,4,68.66,Psalter,43,"We have heard with our ears, O God, for our fa...",False,B,2,...,NaN,7.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Create in me a clean heart,BERT,5,68.31,Bible,147,Alleluia of Aggeus and Zacharias Praise the Lo...,False,B,2,...,NaN,6.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
5,prayer,BERT,1,43.82,Bible,18,For the End a psalm by David The heavens decla...,False,B,0,...,NaN,6.0,NaN,NaN,NaN,NaN,3.0,9.0,NaN,NaN
6,prayer,BERT,2,40.30,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",False,B,1,...,NaN,NaN,NaN,NaN,NaN,7.0,2.0,NaN,NaN,NaN
7,prayer,BERT,3,39.39,Psalter,42,"Judge me, God, and give judgment in my cause a...",False,B,0,...,NaN,8.0,6.0,NaN,NaN,NaN,NaN,10.0,NaN,NaN
8,prayer,BERT,4,38.91,Psalter,149,"Sing unto the Lord a new song, His praise is i...",False,B,3,...,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN
9,prayer,BERT,5,38.70,Psalter,53,"Save me, O God, by Thy name, and judge me by T...",False,B,3,...,NaN,10.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
keys = [
    'Query',
    'Method',
    'Rank',
    'Similarity Score (%)',
    'Text',
    'Psalm Num',
    'Verse'
]

human_scores = human_scores.merge(
    filtered_scores[keys + ['Score']],
    on=keys,
    how='left',
    suffixes=('', '_LLM')
)

human_scores.head(50)